# Notebook 7: Econometric Results and Diagnostics

**Research Project:** Improving Asymmetric Exchange Rate Pass-Through Modelling Across Food Price Categories in South Africa Using Machine Learning

**Research Period:** April 2017 – December 2025

## Notebook Objective

This notebook evaluates the statistical adequacy and reliability of the econometric models developed in Notebook 6.

The analysis focuses on:

1. reproducing the selected ARDL and NARDL specifications
2. assessing residual serial correlation
3. testing for heteroskedasticity
4. examining residual normality
5. evaluating parameter and model stability
6. reviewing models with diagnostic failures
7. consolidating the long-run and short-run findings
8. exporting validated econometric results for later comparison with the machine learning models.

No new lag selection is performed in this notebook. The specifications selected
in Notebook 6 are treated as the starting point for diagnostic assessment.

## Diagnostic Context

Statistically significant coefficients do not necessarily imply that a model is adequately specified.

ARDL and NARDL inference depends on assumptions concerning the behaviour of the model residuals and the stability of the estimated relationship. Diagnostic testing is therefore required before the estimated pass-through effects are treated as final research findings.

The diagnostic process distinguishes between:

- a statistically estimated relationship;
- a relationship that passes the relevant diagnostic tests; and
- a relationship that remains useful but requires a methodological caveat.

This distinction prevents coefficient significance from being interpreted without considering the reliability of the underlying model.

## Diagnostic Framework

The selected econometric models are assessed using the following diagnostic areas:

### Serial correlation

Residual serial correlation indicates that the model has not fully captured the time-dependent structure of the series.

### Heteroskedasticity

Heteroskedastic residuals have non-constant variance and may affect the
reliability of conventional standard errors and hypothesis tests.

### Residual normality

Normality is assessed because strongly non-normal residuals may influence small-sample statistical inference. Normality is treated as a supporting diagnostic rather than an automatic model-rejection rule.

### Functional form

Functional-form assessment examines whether important nonlinear structure may remain unexplained by the model.

### Parameter stability

Stability tests assess whether the estimated relationship remains reasonably consistent across the modelling period.

A 5% significance level is used unless otherwise stated. Diagnostic failures are reported transparently and considered jointly rather than using one test as an automatic reason to discard a model.

In [128]:
# import required libraries
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scipy import stats
from statsmodels.tsa.ardl import ARDL, UECM
from statsmodels.stats.stattools import jarque_bera
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.diagnostic import (
    acorr_ljungbox,
    breaks_cusumolsresid,
    het_arch,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.6f}".format)

print("Libraries imported successfully.")

Libraries imported successfully.


## Load Econometric Data and Results

The category-level econometric dataset and the result tables exported by Notebook 6 are loaded from their established project locations.

The exported lag-selection tables provide the specifications required to re-estimate the models. The remaining tables preserve the bounds-test, error-correction, coefficient and asymmetry findings that will be evaluated against the diagnostic results.

In [129]:
# define input locations
econometric_data_path = Path(
    "../data/processed/econometric_model_data.csv"
)

econometric_results_directory = Path(
    "../reports/tables/econometrics"
)

result_file_names = [
    "symmetric_lag_selection.csv",
    "asymmetric_lag_selection.csv",
    "long_run_bic_comparison.csv",
    "final_bounds_results.csv",
    "error_correction_results.csv",
    "long_run_effects.csv",
    "long_run_asymmetry_results.csv",
    "symmetric_short_run_selection.csv",
    "asymmetric_short_run_selection.csv",
    "short_run_bic_comparison.csv",
    "short_run_asymmetry_results.csv",
]

required_paths = [
    econometric_data_path,
    *[
        econometric_results_directory / file_name
        for file_name in result_file_names
    ],
]

missing_paths = [
    path
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    missing_path_list = "\n".join(
        str(path) for path in missing_paths
    )
    raise FileNotFoundError(
        f"Required Notebook 6 outputs are missing:\n{missing_path_list}"
    )

print("All required input files are available.")

All required input files are available.


In [130]:
# load the econometric dataset
econometric_data = pd.read_csv(
    econometric_data_path,
    parse_dates=["Date"],
)

# Load the exported result tables
econometric_result_tables = {
    Path(file_name).stem: pd.read_csv(
        econometric_results_directory / file_name
    )
    for file_name in result_file_names
}

symmetric_lag_selection = econometric_result_tables[
    "symmetric_lag_selection"
]
asymmetric_lag_selection = econometric_result_tables[
    "asymmetric_lag_selection"
]
final_bounds_results = econometric_result_tables[
    "final_bounds_results"
]
error_correction_results = econometric_result_tables[
    "error_correction_results"
]
long_run_effects = econometric_result_tables[
    "long_run_effects"
]
long_run_asymmetry_results = econometric_result_tables[
    "long_run_asymmetry_results"
]
symmetric_short_run_selection = econometric_result_tables[
    "symmetric_short_run_selection"
]
asymmetric_short_run_selection = econometric_result_tables[
    "asymmetric_short_run_selection"
]
short_run_asymmetry_results = econometric_result_tables[
    "short_run_asymmetry_results"
]

print("Econometric data and result tables loaded successfully.")

Econometric data and result tables loaded successfully.


In [131]:
# validate the Notebook 6 handoff
input_validation = pd.Series(
    {
        "Econometric observations": len(econometric_data),
        "Food subclasses": econometric_data[
            "SubclassDescription"
        ].nunique(),
        "Unique months": econometric_data["Date"].nunique(),
        "Duplicate subclass-month rows": econometric_data.duplicated(
            subset=["SubclassDescription", "Date"]
        ).sum(),
        "Missing econometric values": int(
            econometric_data.isna().sum().sum()
        ),
        "Result tables loaded": len(econometric_result_tables),
        "Symmetric specifications": len(
            symmetric_lag_selection
        ),
        "Asymmetric specifications": len(
            asymmetric_lag_selection
        ),
        "Final bounds-test results": len(
            final_bounds_results
        ),
    },
    name="Value",
).to_frame()

display(input_validation)

result_table_manifest = pd.DataFrame(
    [
        {
            "Table": table_name,
            "Rows": result_table.shape[0],
            "Columns": result_table.shape[1],
        }
        for table_name, result_table
        in econometric_result_tables.items()
    ]
)

display(result_table_manifest)

,Value
Econometric observations,4830
Food subclasses,46
Unique months,105
Duplicate subclass-month rows,0
Missing econometric values,0
Result tables loaded,11
Symmetric specifications,46
Asymmetric specifications,46
Final bounds-test results,92


,Table,Rows,Columns
0,symmetric_lag_selection,46,7
1,asymmetric_lag_selection,46,7
2,long_run_bic_comparison,46,11
3,final_bounds_results,92,12
4,error_correction_results,9,9
5,long_run_effects,14,9
6,long_run_asymmetry_results,6,13
7,symmetric_short_run_selection,46,7
8,asymmetric_short_run_selection,46,7
9,short_run_bic_comparison,46,11


## Re-estimate Selected Models

The selected models are re-estimated using the lag orders exported by Notebook 6.

Four model collections are reconstructed:

1. symmetric ARDL level models
2. asymmetric NARDL level models
3. symmetric stationary short-run models
4. asymmetric stationary short-run models

The same dependent variables, exchange-rate variables, seasonal indicators and six-month hold-back period used during model selection are retained. This ensures that the diagnostic tests are applied to the exact specifications from which the reported results were obtained.

Re-estimation also makes the model residuals and fitted values available within the current notebook without relying on temporary Python objects from Notebook 6.

### Re-estimate the Level Models

The selected symmetric ARDL and asymmetric NARDL level models are reconstructed from their exported lag specifications.

For every food subclass, the reconstruction retains:

- log CPI as the dependent variable
- the selected food-price lag order
- the selected exchange-rate or component lag order
- a constant
- monthly seasonal indicators
- a seasonal period of 12 months
- the common six-month hold-back period

The re-estimated BIC values are compared with the values exported by Notebook 6. Matching BIC values confirm that the same model specifications have been reproduced.

In [132]:
def fit_selected_level_models(
    data,
    selection_table,
    exogenous_columns,
    exchange_rate_lag_column,
):
    """Re-estimate selected level ARDL and UECM models."""

    ardl_results = {}
    uecm_results = {}
    validation_records = []

    for selection_row in selection_table.itertuples(index=False):
        subclass = selection_row.SubclassDescription
        price_lag = int(selection_row.Price_Lag)
        exchange_rate_lag = int(
            getattr(selection_row, exchange_rate_lag_column)
        )

        subclass_data = (
            data.loc[
                data["SubclassDescription"].eq(subclass)
            ]
            .sort_values("Date")
            .set_index("Date")
            .asfreq("MS")
        )

        ardl_model = ARDL(
            endog=subclass_data["Log_CPI"],
            lags=price_lag,
            exog=subclass_data[exogenous_columns],
            order=exchange_rate_lag,
            trend="c",
            seasonal=True,
            period=12,
            causal=False,
            hold_back=6,
            missing="raise",
        )

        ardl_result = ardl_model.fit()
        uecm_result = UECM.from_ardl(
            ardl_result.model
        ).fit()

        ardl_results[subclass] = ardl_result
        uecm_results[subclass] = uecm_result

        validation_records.append(
            {
                "SubclassDescription": subclass,
                "Price_Lag": price_lag,
                "Exchange_Rate_Lag": exchange_rate_lag,
                "Exported_BIC": selection_row.BIC,
                "Reestimated_BIC": ardl_result.bic,
                "Absolute_BIC_Difference": abs(
                    selection_row.BIC - ardl_result.bic
                ),
                "BIC_Match": np.isclose(
                    selection_row.BIC,
                    ardl_result.bic,
                    rtol=1e-10,
                    atol=1e-8,
                ),
                "Observations": int(ardl_result.nobs),
            }
        )

    validation_table = (
        pd.DataFrame(validation_records)
        .sort_values("SubclassDescription")
        .reset_index(drop=True)
    )

    return ardl_results, uecm_results, validation_table

In [133]:
# re-estimate symmetric level models
(
    symmetric_level_ardl_results,
    symmetric_level_uecm_results,
    symmetric_level_validation,
) = fit_selected_level_models(
    data=econometric_data,
    selection_table=symmetric_lag_selection,
    exogenous_columns=["Log_ExchangeRate"],
    exchange_rate_lag_column="Exchange_Rate_Lag",
)

# Re-estimate asymmetric level models
(
    asymmetric_level_ardl_results,
    asymmetric_level_uecm_results,
    asymmetric_level_validation,
) = fit_selected_level_models(
    data=econometric_data,
    selection_table=asymmetric_lag_selection,
    exogenous_columns=[
        "ExchangeRate_Positive_Cumulative_Pct",
        "ExchangeRate_Negative_Cumulative_Pct",
    ],
    exchange_rate_lag_column="Component_Lag",
)

print(
    "Symmetric level models re-estimated:",
    len(symmetric_level_ardl_results),
)
print(
    "Asymmetric level models re-estimated:",
    len(asymmetric_level_ardl_results),
)

Symmetric level models re-estimated: 46
Asymmetric level models re-estimated: 46


In [134]:
# validate the reconstructed level models
symmetric_level_validation["Model"] = "ARDL"
asymmetric_level_validation["Model"] = "NARDL"

level_model_validation = pd.concat(
    [
        symmetric_level_validation,
        asymmetric_level_validation,
    ],
    ignore_index=True,
)

level_reproduction_summary = (
    level_model_validation
    .groupby("Model", observed=True)
    .agg(
        Models=("SubclassDescription", "size"),
        BIC_Matches=("BIC_Match", "sum"),
        Maximum_BIC_Difference=(
            "Absolute_BIC_Difference",
            "max",
        ),
        Minimum_Observations=("Observations", "min"),
        Maximum_Observations=("Observations", "max"),
    )
    .reset_index()
)

display(level_reproduction_summary)

print(
    "All level-model BIC values reproduced:",
    level_model_validation["BIC_Match"].all(),
)

,Model,Models,BIC_Matches,Maximum_BIC_Difference,Minimum_Observations,Maximum_Observations
0,ARDL,46,46,0.000000,99,99
1,NARDL,46,46,0.000000,99,99


All level-model BIC values reproduced: True


### 4.2 Re-estimate the Short-Run Models

The stationary symmetric and asymmetric short-run models are reconstructed from their exported lag specifications.

Monthly food-price inflation is used as the dependent variable. The symmetric models use total monthly exchange-rate changes, while the asymmetric models use separate depreciation and signed appreciation shocks.

Unlike the level models, these specifications do not require a UECM
representation because all variables enter as stationary monthly changes.

In [135]:
def fit_selected_short_run_models(
    data,
    selection_table,
    exogenous_columns,
):
    """Re-estimate selected stationary short-run models."""

    fitted_results = {}
    validation_records = []

    for selection_row in selection_table.itertuples(index=False):
        subclass = selection_row.SubclassDescription
        price_lag = int(selection_row.Price_Lag)
        exchange_rate_lag = int(
            selection_row.Exchange_Rate_Lag
        )

        subclass_data = (
            data.loc[
                data["SubclassDescription"].eq(subclass)
            ]
            .sort_values("Date")
            .set_index("Date")
        )

        exogenous_order = {
            column: exchange_rate_lag
            for column in exogenous_columns
        }

        short_run_model = ARDL(
            endog=subclass_data["Food_Inflation_Pct"],
            lags=price_lag,
            exog=subclass_data[exogenous_columns],
            order=exogenous_order,
            trend="c",
            seasonal=True,
            period=12,
            causal=False,
            hold_back=6,
            missing="raise",
        )

        fitted_result = short_run_model.fit()
        fitted_results[subclass] = fitted_result

        validation_records.append(
            {
                "SubclassDescription": subclass,
                "Price_Lag": price_lag,
                "Exchange_Rate_Lag": exchange_rate_lag,
                "Exported_BIC": selection_row.BIC,
                "Reestimated_BIC": fitted_result.bic,
                "Absolute_BIC_Difference": abs(
                    selection_row.BIC - fitted_result.bic
                ),
                "BIC_Match": np.isclose(
                    selection_row.BIC,
                    fitted_result.bic,
                    rtol=1e-10,
                    atol=1e-8,
                ),
                "Observations": int(fitted_result.nobs),
            }
        )

    validation_table = (
        pd.DataFrame(validation_records)
        .sort_values("SubclassDescription")
        .reset_index(drop=True)
    )

    return fitted_results, validation_table

In [136]:
# re-estimate symmetric short-run models
(
    symmetric_short_run_results,
    symmetric_short_run_validation,
) = fit_selected_short_run_models(
    data=econometric_data,
    selection_table=symmetric_short_run_selection,
    exogenous_columns=["ExchangeRate_Log_Change_Pct"],
)

# Re-estimate asymmetric short-run models
(
    asymmetric_short_run_results,
    asymmetric_short_run_validation,
) = fit_selected_short_run_models(
    data=econometric_data,
    selection_table=asymmetric_short_run_selection,
    exogenous_columns=[
        "Depreciation_Shock_Pct",
        "Appreciation_Shock_Pct",
    ],
)

print(
    "Symmetric short-run models re-estimated:",
    len(symmetric_short_run_results),
)
print(
    "Asymmetric short-run models re-estimated:",
    len(asymmetric_short_run_results),
)

d:\ERPT_ML_Research\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
d:\ERPT_ML_Research\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
d:\ERPT_ML_Research\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
d:\ERPT_ML_Research\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
d:\ERPT_ML_Research\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be use

Symmetric short-run models re-estimated: 46
Asymmetric short-run models re-estimated: 46


d:\ERPT_ML_Research\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
d:\ERPT_ML_Research\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
d:\ERPT_ML_Research\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
d:\ERPT_ML_Research\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
d:\ERPT_ML_Research\.venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be use

In [137]:
# validate the reconstructed short-run models
symmetric_short_run_validation["Model"] = "Symmetric"
asymmetric_short_run_validation["Model"] = "Asymmetric"

short_run_model_validation = pd.concat(
    [
        symmetric_short_run_validation,
        asymmetric_short_run_validation,
    ],
    ignore_index=True,
)

short_run_reproduction_summary = (
    short_run_model_validation
    .groupby("Model", observed=True)
    .agg(
        Models=("SubclassDescription", "size"),
        BIC_Matches=("BIC_Match", "sum"),
        Maximum_BIC_Difference=(
            "Absolute_BIC_Difference",
            "max",
        ),
        Minimum_Observations=("Observations", "min"),
        Maximum_Observations=("Observations", "max"),
    )
    .reset_index()
)

display(short_run_reproduction_summary)

print(
    "All short-run BIC values reproduced:",
    short_run_model_validation["BIC_Match"].all(),
)

,Model,Models,BIC_Matches,Maximum_BIC_Difference,Minimum_Observations,Maximum_Observations
0,Asymmetric,46,46,0.000000,99,99
1,Symmetric,46,46,0.000000,99,99


All short-run BIC values reproduced: True


## Residual Serial-Correlation Diagnostics

Residual serial correlation occurs when model errors remain related across time. Its presence may indicate that the selected lag structure has not fully captured the dynamics in food-price inflation.

The Ljung–Box test is applied at 12 lags because the data are monthly. The test therefore assesses whether the residual autocorrelations through one annual cycle are jointly equal to zero.

The hypotheses are:

- **Null hypothesis:** The residuals are not serially correlated.
- **Alternative hypothesis:** The residuals exhibit serial correlation.

A p-value below 0.05 is treated as evidence against the null hypothesis.

The diagnostic is applied to the selected symmetric and asymmetric level models, as well as the corresponding short-run models. The level-model diagnostics use the ARDL representations because their UECM forms are reparameterisations of the same underlying models.

In [138]:
# Configure the Serial-Correlation Tests

serial_correlation_lags = 12

diagnostic_model_collections = {
    "Symmetric level": symmetric_level_ardl_results,
    "Asymmetric level": asymmetric_level_ardl_results,
    "Symmetric short run": symmetric_short_run_results,
    "Asymmetric short run": asymmetric_short_run_results,
}

model_collection_summary = pd.DataFrame(
    {
        "Model": diagnostic_model_collections.keys(),
        "Models": [
            len(model_results)
            for model_results in diagnostic_model_collections.values()
        ],
    }
)

display(model_collection_summary)

,Model,Models
0,Symmetric level,46
1,Asymmetric level,46
2,Symmetric short run,46
3,Asymmetric short run,46


In [139]:
# run the Ljung–Box Tests

def run_serial_correlation_tests(model_collections, test_lag=12):
    records = []

    for model_name, model_results in model_collections.items():
        for subclass, fitted_model in model_results.items():
            residuals = np.asarray(fitted_model.resid, dtype=float)
            residuals = residuals[np.isfinite(residuals)]

            ar_lags = getattr(fitted_model.model, "ar_lags", None)
            model_df = len(ar_lags) if ar_lags is not None else 0

            test_output = acorr_ljungbox(
                residuals,
                lags=[test_lag],
                model_df=model_df,
                return_df=True,
            )

            selected_result = test_output.iloc[0]
            test_statistic = float(selected_result["lb_stat"])
            p_value = float(selected_result["lb_pvalue"])

            records.append(
                {
                    "SubclassDescription": subclass,
                    "Model": model_name,
                    "Residual_Observations": len(residuals),
                    "AR_Lags": model_df,
                    "Test_Lag": test_lag,
                    "Degrees_of_Freedom": test_lag - model_df,
                    "Ljung_Box_Statistic": test_statistic,
                    "P_Value": p_value,
                    "No_Serial_Correlation": p_value >= 0.05,
                }
            )

    return pd.DataFrame(records)


serial_correlation_results = run_serial_correlation_tests(
    diagnostic_model_collections,
    test_lag=serial_correlation_lags,
)

print(
    "Serial-correlation tests completed:",
    len(serial_correlation_results),
)

print(
    "Missing diagnostic p-values:",
    serial_correlation_results["P_Value"].isna().sum(),
)

Serial-correlation tests completed: 184
Missing diagnostic p-values: 0


In [140]:
# summarise the Diagnostic Results

serial_correlation_summary = (
    serial_correlation_results
    .groupby("Model", as_index=False)
    .agg(
        Models=("SubclassDescription", "size"),
        Passed=("No_Serial_Correlation", "sum"),
    )
)

serial_correlation_summary["Failed"] = (
    serial_correlation_summary["Models"]
    - serial_correlation_summary["Passed"]
)

serial_correlation_summary["Pass_Rate_Pct"] = (
    100
    * serial_correlation_summary["Passed"]
    / serial_correlation_summary["Models"]
)

display(serial_correlation_summary)

,Model,Models,Passed,Failed,Pass_Rate_Pct
0,Asymmetric level,46,37,9,80.434783
1,Asymmetric short run,46,40,6,86.956522
2,Symmetric level,46,37,9,80.434783
3,Symmetric short run,46,41,5,89.130435


In [141]:
# identify Models with Serial Correlation

serial_correlation_failures = (
    serial_correlation_results.loc[
        ~serial_correlation_results["No_Serial_Correlation"]
    ]
    .sort_values(["Model", "P_Value"])
    .reset_index(drop=True)
)

print(
    "Models with evidence of serial correlation:",
    len(serial_correlation_failures),
)

if serial_correlation_failures.empty:
    print("No models failed the Ljung–Box test at the 5% level.")
else:
    display(serial_correlation_failures)

Models with evidence of serial correlation: 29


,SubclassDescription,Model,Residual_Observations,AR_Lags,Test_Lag,Degrees_of_Freedom,Ljung_Box_Statistic,P_Value,No_Serial_Correlation
0,"Macaroni, noodles, couscous and similar pasta ...",Asymmetric level,99,1,12,11,28.208724,0.003007,False
1,Fruit and vegetable juices,Asymmetric level,99,1,12,11,26.194036,0.006072,False
2,Baby food,Asymmetric level,99,2,12,10,24.332343,0.006765,False
3,Tubers,Asymmetric level,99,2,12,10,23.809259,0.008123,False
4,Milk,Asymmetric level,99,1,12,11,22.031833,0.024128,False
5,Other milk and cream,Asymmetric level,99,1,12,11,21.827834,0.025741,False
6,Coffee and coffee substitutes,Asymmetric level,99,1,12,11,21.659422,0.027149,False
7,Other non-alcoholic beverages,Asymmetric level,99,1,12,11,21.632521,0.027380,False
8,Other food products n.e.c.,Asymmetric level,99,2,12,10,18.517395,0.046838,False
9,"Macaroni, noodles, couscous and similar pasta ...",Asymmetric short run,99,3,12,9,28.162237,0.000896,False


### Serial-Correlation Results

Most selected models do not exhibit significant residual serial correlation at the 5% level. The pass rates range from 80.4% for the level specifications to 89.1% for the symmetric short-run specifications.

The diagnostic failures are concentrated in particular food subclasses rather than affecting all models uniformly. Several subclasses fail under more than one specification, indicating that some category-specific price dynamics may not be fully captured by the BIC-selected lag structures.

These failures do not invalidate the complete econometric analysis. However, coefficient inference from affected specifications should be treated cautiously because residual dependence can affect conventional standard errors and hypothesis tests.

The most important next check is whether the cointegrated level models used for the primary long-run results satisfy the serial-correlation diagnostic.

In [142]:
# check the Primary Long-Run Models

level_serial_correlation = serial_correlation_results.loc[
    serial_correlation_results["Model"].isin(
        ["Symmetric level", "Asymmetric level"]
    )
].copy()

level_serial_correlation["Model"] = (
    level_serial_correlation["Model"]
    .replace(
        {
            "Symmetric level": "ARDL",
            "Asymmetric level": "NARDL",
        }
    )
)

primary_long_run_models = final_bounds_results.loc[
    final_bounds_results["Final_Decision"].eq("Cointegration"),
    [
        "SubclassDescription",
        "Model",
        "Bounds_Statistic",
        "Final_Decision",
    ],
].copy()

primary_long_run_serial_diagnostics = primary_long_run_models.merge(
    level_serial_correlation[
        [
            "SubclassDescription",
            "Model",
            "Ljung_Box_Statistic",
            "P_Value",
            "No_Serial_Correlation",
        ]
    ],
    on=["SubclassDescription", "Model"],
    how="left",
    validate="one_to_one",
)

display(
    primary_long_run_serial_diagnostics.sort_values(
        ["Model", "P_Value"]
    ).reset_index(drop=True)
)

primary_long_run_serial_summary = (
    primary_long_run_serial_diagnostics
    .groupby("Model", as_index=False)
    .agg(
        Models=("SubclassDescription", "size"),
        Passed=("No_Serial_Correlation", "sum"),
    )
)

primary_long_run_serial_summary["Failed"] = (
    primary_long_run_serial_summary["Models"]
    - primary_long_run_serial_summary["Passed"]
)

display(primary_long_run_serial_summary)

,SubclassDescription,Model,Bounds_Statistic,Final_Decision,Ljung_Box_Statistic,P_Value,No_Serial_Correlation
0,"Chocolate, cocoa, and cocoa-based food products",ARDL,7.502411,Cointegration,16.192063,0.134149,True
1,Yoghurt and similar products,ARDL,7.295599,Cointegration,11.420135,0.325735,True
2,"Other vegetables, fresh or chilled",ARDL,7.372386,Cointegration,7.387174,0.688457,True
3,"Fruit-bearing vegetables, fresh or chilled",NARDL,8.346412,Cointegration,16.665141,0.054225,True
4,"Chocolate, cocoa, and cocoa-based food products",NARDL,6.235160,Cointegration,17.906933,0.083765,True
5,Cereals,NARDL,6.620570,Cointegration,17.180201,0.102657,True
6,"Dates, figs and tropical fruits, fresh",NARDL,9.173770,Cointegration,12.717140,0.239920,True
7,Yoghurt and similar products,NARDL,7.181472,Cointegration,12.577165,0.248283,True
8,"Other vegetables, fresh or chilled",NARDL,7.406674,Cointegration,6.700292,0.753404,True


,Model,Models,Passed,Failed
0,ARDL,3,3,0
1,NARDL,6,6,0


### Primary Long-Run Model Assessment

All nine bounds-supported level models pass the Ljung–Box test at the 5% significance level. This includes all three ARDL specifications and all six NARDL specifications retained by the sample-specific bounds tests.

The NARDL model for fruit-bearing vegetables has a p-value of 0.054, which is close to the 5% threshold. It is therefore retained but treated as a borderline diagnostic result.

Overall, the absence of significant residual serial correlation supports the dynamic specification of the models used for the primary long-run analysis. This finding does not override the separate error-correction assessment. In particular, the chocolate ARDL specification remains unsuitable for primary long-run interpretation because its adjustment coefficient was not statistically significant.

In [143]:
# run the ARCH-LM Tests

def run_arch_tests(model_collections, test_lag=12):
    records = []

    for model_name, model_results in model_collections.items():
        for subclass, fitted_model in model_results.items():
            residuals = np.asarray(fitted_model.resid, dtype=float)
            residuals = residuals[np.isfinite(residuals)]

            lm_statistic, lm_p_value, f_statistic, f_p_value = het_arch(
                residuals,
                nlags=test_lag,
            )

            records.append(
                {
                    "SubclassDescription": subclass,
                    "Model": model_name,
                    "Residual_Observations": len(residuals),
                    "Test_Lag": test_lag,
                    "ARCH_LM_Statistic": float(lm_statistic),
                    "ARCH_LM_P_Value": float(lm_p_value),
                    "F_Statistic": float(f_statistic),
                    "F_P_Value": float(f_p_value),
                    "No_ARCH_Effects": lm_p_value >= 0.05,
                }
            )

    return pd.DataFrame(records)


arch_test_lags = 12

heteroskedasticity_results = run_arch_tests(
    diagnostic_model_collections,
    test_lag=arch_test_lags,
)

print(
    "ARCH-LM tests completed:",
    len(heteroskedasticity_results),
)

print(
    "Missing ARCH-LM p-values:",
    heteroskedasticity_results["ARCH_LM_P_Value"].isna().sum(),
)

ARCH-LM tests completed: 184
Missing ARCH-LM p-values: 0


In [144]:
# summarise the ARCH-LM Results

heteroskedasticity_summary = (
    heteroskedasticity_results
    .groupby("Model", as_index=False)
    .agg(
        Models=("SubclassDescription", "size"),
        Passed=("No_ARCH_Effects", "sum"),
    )
)

heteroskedasticity_summary["Failed"] = (
    heteroskedasticity_summary["Models"]
    - heteroskedasticity_summary["Passed"]
)

heteroskedasticity_summary["Pass_Rate_Pct"] = (
    100
    * heteroskedasticity_summary["Passed"]
    / heteroskedasticity_summary["Models"]
)

display(heteroskedasticity_summary)

,Model,Models,Passed,Failed,Pass_Rate_Pct
0,Asymmetric level,46,42,4,91.304348
1,Asymmetric short run,46,42,4,91.304348
2,Symmetric level,46,43,3,93.478261
3,Symmetric short run,46,44,2,95.652174


In [145]:
# identify Models with ARCH Effects

heteroskedasticity_failures = (
    heteroskedasticity_results.loc[
        ~heteroskedasticity_results["No_ARCH_Effects"]
    ]
    .sort_values(["Model", "ARCH_LM_P_Value"])
    .reset_index(drop=True)
)

print(
    "Models with evidence of ARCH effects:",
    len(heteroskedasticity_failures),
)

if heteroskedasticity_failures.empty:
    print("No models exhibit ARCH effects at the 5% level.")
else:
    display(heteroskedasticity_failures)

Models with evidence of ARCH effects: 13


,SubclassDescription,Model,Residual_Observations,Test_Lag,ARCH_LM_Statistic,ARCH_LM_P_Value,F_Statistic,F_P_Value,No_ARCH_Effects
0,Vegetable oils,Asymmetric level,99,12,28.696691,0.004369,3.035212,0.001638,False
1,"Meat, fresh, chilled or frozen",Asymmetric level,99,12,28.243502,0.005096,2.964238,0.002030,False
2,Margarine and similar preparations,Asymmetric level,99,12,25.750050,0.011642,2.592524,0.006228,False
3,Milk,Asymmetric level,99,12,24.515586,0.017293,2.419475,0.010464,False
4,"Other vegetables, fresh or chilled",Asymmetric short run,99,12,31.363529,0.001733,3.476289,0.000435,False
5,Vegetable oils,Asymmetric short run,99,12,27.989182,0.005552,2.924887,0.002286,False
6,Milk,Asymmetric short run,99,12,25.841600,0.011301,2.605636,0.005987,False
7,"Meat, fresh, chilled or frozen",Asymmetric short run,99,12,21.183363,0.047759,1.984768,0.037623,False
8,Milk,Symmetric level,99,12,28.960255,0.003994,3.076999,0.001444,False
9,Vegetable oils,Symmetric level,99,12,27.773504,0.005970,2.891779,0.002526,False


### Conditional Heteroskedasticity Results

Most selected specifications do not exhibit significant ARCH effects. Pass rates exceed 91% across all four model families, with the symmetric short-run models recording the highest pass rate.

The 13 diagnostic failures represent model–subclass combinations rather than 13 different food subclasses. Evidence of volatility clustering is concentrated in a small group of categories, including vegetable oils, milk, meat and other vegetables.

These results suggest that conditional heteroskedasticity is category-specific rather than a general feature of the econometric models. Conventional inference from affected specifications may nevertheless require additional caution or robust standard errors.

The primary long-run models are assessed separately because they provide the main econometric evidence used in the research findings.

In [146]:
# check the Primary Long-Run Models

level_heteroskedasticity_results = heteroskedasticity_results.loc[
    heteroskedasticity_results["Model"].isin(
        ["Symmetric level", "Asymmetric level"]
    )
].copy()

level_heteroskedasticity_results["Model"] = (
    level_heteroskedasticity_results["Model"]
    .replace(
        {
            "Symmetric level": "ARDL",
            "Asymmetric level": "NARDL",
        }
    )
)

primary_long_run_arch_diagnostics = primary_long_run_models.merge(
    level_heteroskedasticity_results[
        [
            "SubclassDescription",
            "Model",
            "ARCH_LM_Statistic",
            "ARCH_LM_P_Value",
            "No_ARCH_Effects",
        ]
    ],
    on=["SubclassDescription", "Model"],
    how="left",
    validate="one_to_one",
)

display(
    primary_long_run_arch_diagnostics.sort_values(
        ["Model", "ARCH_LM_P_Value"]
    ).reset_index(drop=True)
)

primary_long_run_arch_summary = (
    primary_long_run_arch_diagnostics
    .groupby("Model", as_index=False)
    .agg(
        Models=("SubclassDescription", "size"),
        Passed=("No_ARCH_Effects", "sum"),
    )
)

primary_long_run_arch_summary["Failed"] = (
    primary_long_run_arch_summary["Models"]
    - primary_long_run_arch_summary["Passed"]
)

display(primary_long_run_arch_summary)

,SubclassDescription,Model,Bounds_Statistic,Final_Decision,ARCH_LM_Statistic,ARCH_LM_P_Value,No_ARCH_Effects
0,"Other vegetables, fresh or chilled",ARDL,7.372386,Cointegration,17.008230,0.149288,True
1,"Chocolate, cocoa, and cocoa-based food products",ARDL,7.502411,Cointegration,9.638326,0.647656,True
2,Yoghurt and similar products,ARDL,7.295599,Cointegration,8.259382,0.764546,True
3,"Other vegetables, fresh or chilled",NARDL,7.406674,Cointegration,20.815796,0.053144,True
4,Cereals,NARDL,6.620570,Cointegration,10.066129,0.610159,True
5,"Chocolate, cocoa, and cocoa-based food products",NARDL,6.235160,Cointegration,9.562705,0.654264,True
6,Yoghurt and similar products,NARDL,7.181472,Cointegration,9.064428,0.697418,True
7,"Dates, figs and tropical fruits, fresh",NARDL,9.173770,Cointegration,7.430630,0.827894,True
8,"Fruit-bearing vegetables, fresh or chilled",NARDL,8.346412,Cointegration,7.140306,0.848204,True


,Model,Models,Passed,Failed
0,ARDL,3,3,0
1,NARDL,6,6,0


### Primary Long-Run Model Assessment

All nine bounds-supported level models pass the ARCH-LM test at the 5%
significance level. The primary long-run specifications therefore show no statistically significant evidence of conditional heteroskedasticity.

The NARDL model for other vegetables has a p-value of 0.053, which is close to the decision threshold. This model is retained, but its variance diagnostic is treated as borderline.

Combined with the serial-correlation results, the primary long-run model set shows generally well-behaved residual dependence and variance. The separate error-correction requirement remains applicable when determining which models support substantive long-run interpretation.

In [147]:
# run the Jarque–Bera Tests

def run_normality_tests(model_collections):
    records = []

    for model_name, model_results in model_collections.items():
        for subclass, fitted_model in model_results.items():
            residuals = np.asarray(fitted_model.resid, dtype=float)
            residuals = residuals[np.isfinite(residuals)]

            jb_statistic, p_value, skewness, kurtosis = jarque_bera(
                residuals
            )

            records.append(
                {
                    "SubclassDescription": subclass,
                    "Model": model_name,
                    "Residual_Observations": len(residuals),
                    "Jarque_Bera_Statistic": float(jb_statistic),
                    "P_Value": float(p_value),
                    "Skewness": float(skewness),
                    "Kurtosis": float(kurtosis),
                    "Normal_Residuals": p_value >= 0.05,
                }
            )

    return pd.DataFrame(records)


normality_results = run_normality_tests(
    diagnostic_model_collections
)

print(
    "Normality tests completed:",
    len(normality_results),
)

print(
    "Missing normality p-values:",
    normality_results["P_Value"].isna().sum(),
)

Normality tests completed: 184
Missing normality p-values: 0


In [148]:
# summarise the normality results

normality_summary = (
    normality_results
    .groupby("Model", as_index=False)
    .agg(
        Models=("SubclassDescription", "size"),
        Passed=("Normal_Residuals", "sum"),
    )
)

normality_summary["Failed"] = (
    normality_summary["Models"]
    - normality_summary["Passed"]
)

normality_summary["Pass_Rate_Pct"] = (
    100
    * normality_summary["Passed"]
    / normality_summary["Models"]
)

display(normality_summary)

,Model,Models,Passed,Failed,Pass_Rate_Pct
0,Asymmetric level,46,23,23,50.000000
1,Asymmetric short run,46,21,25,45.652174
2,Symmetric level,46,23,23,50.000000
3,Symmetric short run,46,22,24,47.826087


In [149]:
# identify Non-Normal Residuals

normality_failures = (
    normality_results.loc[
        ~normality_results["Normal_Residuals"]
    ]
    .sort_values(["Model", "P_Value"])
    .reset_index(drop=True)
)

print(
    "Models rejecting residual normality:",
    len(normality_failures),
)

if normality_failures.empty:
    print("All models pass the Jarque–Bera test.")
else:
    display(normality_failures.head(20))

Models rejecting residual normality: 95


,SubclassDescription,Model,Residual_Observations,Jarque_Bera_Statistic,P_Value,Skewness,Kurtosis,Normal_Residuals
0,Eggs,Asymmetric level,99,906.163792,0.000000,2.104515,17.211268,False
1,"Other sugar, confectionery and dessert",Asymmetric level,99,577.001772,0.000000,1.910675,14.192698,False
2,Baby food,Asymmetric level,99,471.570338,0.000000,1.776368,13.084550,False
3,Fruit and vegetable juices,Asymmetric level,99,310.889878,0.000000,-1.262061,11.306386,False
4,"Meat, offal, blood and other parts of slaughte...",Asymmetric level,99,286.383022,0.000000,1.454468,10.807963,False
5,Sugar,Asymmetric level,99,222.089990,0.000000,1.493131,9.702405,False
6,"Offal, blood and other parts of slaughtered an...",Asymmetric level,99,159.112911,0.000000,1.363448,8.580041,False
7,Soft drinks,Asymmetric level,99,108.132880,0.000000,1.484731,7.170890,False
8,Vegetable oils,Asymmetric level,99,101.601898,0.000000,0.903034,7.622648,False
9,Breakfast cereals,Asymmetric level,99,57.221326,0.000000,-0.100252,6.719091,False


### Residual Normality Results

Residual normality is the weakest diagnostic across the four model families. Approximately half of the selected models reject the null hypothesis of normal residuals at the 5% significance level.

The failures are associated with residual skewness and excess kurtosis. Several food categories display strongly right-skewed residuals, while some have kurtosis substantially above the normal-distribution value of three. This indicates heavy tails and occasional large food-price movements that are not fully represented by a normal error distribution.

Non-normal residuals do not automatically invalidate the estimated
coefficients. However, they can weaken conventional small-sample inference, particularly confidence intervals and hypothesis tests that rely on normality.

The normality of the bounds-supported long-run models is assessed separately before determining whether additional robust inference is required.

In [150]:
# check the Primary Long-Run Models

level_normality_results = normality_results.loc[
    normality_results["Model"].isin(
        ["Symmetric level", "Asymmetric level"]
    )
].copy()

level_normality_results["Model"] = (
    level_normality_results["Model"]
    .replace(
        {
            "Symmetric level": "ARDL",
            "Asymmetric level": "NARDL",
        }
    )
)

primary_long_run_normality_diagnostics = primary_long_run_models.merge(
    level_normality_results[
        [
            "SubclassDescription",
            "Model",
            "Jarque_Bera_Statistic",
            "P_Value",
            "Skewness",
            "Kurtosis",
            "Normal_Residuals",
        ]
    ],
    on=["SubclassDescription", "Model"],
    how="left",
    validate="one_to_one",
)

display(
    primary_long_run_normality_diagnostics.sort_values(
        ["Model", "P_Value"]
    ).reset_index(drop=True)
)

primary_long_run_normality_summary = (
    primary_long_run_normality_diagnostics
    .groupby("Model", as_index=False)
    .agg(
        Models=("SubclassDescription", "size"),
        Passed=("Normal_Residuals", "sum"),
    )
)

primary_long_run_normality_summary["Failed"] = (
    primary_long_run_normality_summary["Models"]
    - primary_long_run_normality_summary["Passed"]
)

display(primary_long_run_normality_summary)

,SubclassDescription,Model,Bounds_Statistic,Final_Decision,Jarque_Bera_Statistic,P_Value,Skewness,Kurtosis,Normal_Residuals
0,Yoghurt and similar products,ARDL,7.295599,Cointegration,50.676504,0.000000,0.968390,5.921317,False
1,"Other vegetables, fresh or chilled",ARDL,7.372386,Cointegration,28.220733,0.000001,0.459039,5.449188,False
2,"Chocolate, cocoa, and cocoa-based food products",ARDL,7.502411,Cointegration,0.938101,0.625596,0.127540,3.402930,True
3,Yoghurt and similar products,NARDL,7.181472,Cointegration,32.232495,0.000000,0.865928,5.194222,False
4,"Other vegetables, fresh or chilled",NARDL,7.406674,Cointegration,28.925244,0.000001,0.651701,5.305065,False
5,"Dates, figs and tropical fruits, fresh",NARDL,9.173770,Cointegration,1.840323,0.398455,-0.303369,3.279299,True
6,Cereals,NARDL,6.620570,Cointegration,1.665720,0.434804,-0.313010,2.890871,True
7,"Chocolate, cocoa, and cocoa-based food products",NARDL,6.235160,Cointegration,1.380112,0.501548,0.237070,3.331305,True
8,"Fruit-bearing vegetables, fresh or chilled",NARDL,8.346412,Cointegration,0.803426,0.669173,0.064907,2.578196,True


,Model,Models,Passed,Failed
0,ARDL,3,1,2
1,NARDL,6,4,2


### Primary Long-Run Model Assessment

Five of the nine bounds-supported models pass the Jarque–Bera normality test. Residual non-normality is concentrated in yoghurt and other vegetables, which reject normality under both the ARDL and NARDL specifications.

These residuals are positively skewed and have kurtosis above five, indicating right-skewed, heavy-tailed disturbances. This may reflect occasional large food-price movements that are not well represented by a normal error distribution.

The cointegration findings are retained because residual normality is not the condition used to establish the existence of a long-run relationship. Nevertheless, conventional coefficient p-values and confidence intervals for the affected models should be interpreted cautiously and assessed using robust inference.

The remaining primary models, including cereals, chocolate, dates and
fruit-bearing vegetables, do not reject residual normality.

In [151]:
# run the CUSUM Stability Tests

def run_cusum_tests(model_collections):
    records = []

    for model_name, model_results in model_collections.items():
        for subclass, fitted_model in model_results.items():
            residuals = np.asarray(fitted_model.resid, dtype=float)
            residuals = residuals[np.isfinite(residuals)]

            parameter_count = int(fitted_model.df_model)

            test_statistic, p_value, critical_values = (
                breaks_cusumolsresid(
                    residuals,
                    ddof=parameter_count,
                )
            )

            critical_value_5pct = next(
                float(value)
                for level, value in critical_values
                if float(level) == 5.0
            )

            records.append(
                {
                    "SubclassDescription": subclass,
                    "Model": model_name,
                    "Residual_Observations": len(residuals),
                    "Parameters": parameter_count,
                    "CUSUM_Statistic": float(test_statistic),
                    "Critical_Value_5pct": critical_value_5pct,
                    "P_Value": float(p_value),
                    "Stable_Parameters": p_value >= 0.05,
                }
            )

    return pd.DataFrame(records)


cusum_results = run_cusum_tests(
    diagnostic_model_collections
)

print("CUSUM tests completed:", len(cusum_results))
print(
    "Missing CUSUM p-values:",
    cusum_results["P_Value"].isna().sum(),
)

CUSUM tests completed: 184
Missing CUSUM p-values: 0


In [152]:
# summarise the Stability Results

cusum_summary = (
    cusum_results
    .groupby("Model", as_index=False)
    .agg(
        Models=("SubclassDescription", "size"),
        Stable=("Stable_Parameters", "sum"),
    )
)

cusum_summary["Unstable"] = (
    cusum_summary["Models"]
    - cusum_summary["Stable"]
)

cusum_summary["Stability_Rate_Pct"] = (
    100
    * cusum_summary["Stable"]
    / cusum_summary["Models"]
)

display(cusum_summary)

,Model,Models,Stable,Unstable,Stability_Rate_Pct
0,Asymmetric level,46,46,0,100.000000
1,Asymmetric short run,46,46,0,100.000000
2,Symmetric level,46,46,0,100.000000
3,Symmetric short run,46,46,0,100.000000


In [153]:
# identify Unstable Specifications

cusum_failures = (
    cusum_results.loc[
        ~cusum_results["Stable_Parameters"]
    ]
    .sort_values(["Model", "P_Value"])
    .reset_index(drop=True)
)

print(
    "Models with evidence of parameter instability:",
    len(cusum_failures),
)

if cusum_failures.empty:
    print("All models pass the CUSUM stability test.")
else:
    display(cusum_failures.head(20))

Models with evidence of parameter instability: 0
All models pass the CUSUM stability test.


### Parameter Stability Results

All 184 selected specifications pass the CUSUM stability test at the 5%
significance level. No model family or food subclass provides statistically significant evidence of parameter instability over the estimation period.

This supports the use of the selected models across the common sample from October 2017 to December 2025. The result suggests that the estimated relationships are not dominated by an abrupt structural change during one particular part of the sample.

The CUSUM result should nevertheless be interpreted as an absence of detected instability rather than proof that the coefficients remained perfectly constant. Like any statistical test, its ability to detect small or gradual changes is limited by the available sample.

In [154]:
# consolidate the Diagnostic Results

serial_diagnostics = (
    serial_correlation_results[
        [
            "SubclassDescription",
            "Model",
            "P_Value",
            "No_Serial_Correlation",
        ]
    ]
    .rename(
        columns={
            "P_Value": "Serial_Correlation_P_Value",
        }
    )
)

arch_diagnostics = heteroskedasticity_results[
    [
        "SubclassDescription",
        "Model",
        "ARCH_LM_P_Value",
        "No_ARCH_Effects",
    ]
]

normality_diagnostics = (
    normality_results[
        [
            "SubclassDescription",
            "Model",
            "P_Value",
            "Skewness",
            "Kurtosis",
            "Normal_Residuals",
        ]
    ]
    .rename(
        columns={
            "P_Value": "Normality_P_Value",
        }
    )
)

stability_diagnostics = (
    cusum_results[
        [
            "SubclassDescription",
            "Model",
            "P_Value",
            "Stable_Parameters",
        ]
    ]
    .rename(
        columns={
            "P_Value": "CUSUM_P_Value",
        }
    )
)

combined_diagnostic_results = (
    serial_diagnostics
    .merge(
        arch_diagnostics,
        on=["SubclassDescription", "Model"],
        validate="one_to_one",
    )
    .merge(
        normality_diagnostics,
        on=["SubclassDescription", "Model"],
        validate="one_to_one",
    )
    .merge(
        stability_diagnostics,
        on=["SubclassDescription", "Model"],
        validate="one_to_one",
    )
)

combined_diagnostic_results["Passed_Core_Diagnostics"] = (
    combined_diagnostic_results["No_Serial_Correlation"]
    & combined_diagnostic_results["No_ARCH_Effects"]
    & combined_diagnostic_results["Stable_Parameters"]
)

combined_diagnostic_results["Passed_All_Diagnostics"] = (
    combined_diagnostic_results["Passed_Core_Diagnostics"]
    & combined_diagnostic_results["Normal_Residuals"]
)

print(
    "Combined diagnostic results:",
    len(combined_diagnostic_results),
)

print(
    "Duplicate model-subclass rows:",
    combined_diagnostic_results.duplicated(
        ["SubclassDescription", "Model"]
    ).sum(),
)

print(
    "Missing diagnostic values:",
    combined_diagnostic_results.isna().sum().sum(),
)

Combined diagnostic results: 184
Duplicate model-subclass rows: 0
Missing diagnostic values: 0


In [155]:
# summarise the Combined Diagnostics

combined_diagnostic_summary = (
    combined_diagnostic_results
    .groupby("Model", as_index=False)
    .agg(
        Models=("SubclassDescription", "size"),
        No_Serial_Correlation=(
            "No_Serial_Correlation",
            "sum",
        ),
        No_ARCH_Effects=("No_ARCH_Effects", "sum"),
        Normal_Residuals=("Normal_Residuals", "sum"),
        Stable_Parameters=("Stable_Parameters", "sum"),
        Core_Diagnostics_Passed=(
            "Passed_Core_Diagnostics",
            "sum",
        ),
        All_Diagnostics_Passed=(
            "Passed_All_Diagnostics",
            "sum",
        ),
    )
)

display(combined_diagnostic_summary)

,Model,Models,No_Serial_Correlation,No_ARCH_Effects,Normal_Residuals,Stable_Parameters,Core_Diagnostics_Passed,All_Diagnostics_Passed
0,Asymmetric level,46,37,42,23,46,34,20
1,Asymmetric short run,46,40,42,21,46,36,18
2,Symmetric level,46,37,43,23,46,35,18
3,Symmetric short run,46,41,44,22,46,39,20


In [156]:
# summarise the Primary Long-Run Models

level_diagnostic_results = combined_diagnostic_results.loc[
    combined_diagnostic_results["Model"].isin(
        ["Symmetric level", "Asymmetric level"]
    )
].copy()

level_diagnostic_results["Model"] = (
    level_diagnostic_results["Model"]
    .replace(
        {
            "Symmetric level": "ARDL",
            "Asymmetric level": "NARDL",
        }
    )
)

primary_model_diagnostics = primary_long_run_models.merge(
    level_diagnostic_results,
    on=["SubclassDescription", "Model"],
    how="left",
    validate="one_to_one",
)

primary_diagnostic_columns = [
    "SubclassDescription",
    "Model",
    "No_Serial_Correlation",
    "No_ARCH_Effects",
    "Normal_Residuals",
    "Stable_Parameters",
    "Passed_Core_Diagnostics",
    "Passed_All_Diagnostics",
]

display(
    primary_model_diagnostics[
        primary_diagnostic_columns
    ].sort_values(
        ["Model", "SubclassDescription"]
    ).reset_index(drop=True)
)

primary_diagnostic_summary = pd.Series(
    {
        "Primary models": len(primary_model_diagnostics),
        "Passed serial-correlation test": (
            primary_model_diagnostics[
                "No_Serial_Correlation"
            ].sum()
        ),
        "Passed ARCH-LM test": (
            primary_model_diagnostics[
                "No_ARCH_Effects"
            ].sum()
        ),
        "Passed normality test": (
            primary_model_diagnostics[
                "Normal_Residuals"
            ].sum()
        ),
        "Passed CUSUM test": (
            primary_model_diagnostics[
                "Stable_Parameters"
            ].sum()
        ),
        "Passed core diagnostics": (
            primary_model_diagnostics[
                "Passed_Core_Diagnostics"
            ].sum()
        ),
        "Passed all diagnostics": (
            primary_model_diagnostics[
                "Passed_All_Diagnostics"
            ].sum()
        ),
    },
    name="Value",
).to_frame()

display(primary_diagnostic_summary)

,SubclassDescription,Model,No_Serial_Correlation,No_ARCH_Effects,Normal_Residuals,Stable_Parameters,Passed_Core_Diagnostics,Passed_All_Diagnostics
0,"Chocolate, cocoa, and cocoa-based food products",ARDL,True,True,True,True,True,True
1,"Other vegetables, fresh or chilled",ARDL,True,True,False,True,True,False
2,Yoghurt and similar products,ARDL,True,True,False,True,True,False
3,Cereals,NARDL,True,True,True,True,True,True
4,"Chocolate, cocoa, and cocoa-based food products",NARDL,True,True,True,True,True,True
5,"Dates, figs and tropical fruits, fresh",NARDL,True,True,True,True,True,True
6,"Fruit-bearing vegetables, fresh or chilled",NARDL,True,True,True,True,True,True
7,"Other vegetables, fresh or chilled",NARDL,True,True,False,True,True,False
8,Yoghurt and similar products,NARDL,True,True,False,True,True,False


,Value
Primary models,9
Passed serial-correlation test,9
Passed ARCH-LM test,9
Passed normality test,5
Passed CUSUM test,9
Passed core diagnostics,9
Passed all diagnostics,5


## Robust Inference

The combined diagnostics show that all nine primary long-run models pass the serial-correlation, ARCH and parameter-stability tests. Residual non-normality is the only diagnostic concern and is concentrated in yoghurt and other vegetables.

The selected model specifications are retained because the core dynamic and stability diagnostics are satisfied. However, the coefficient inference is reassessed using heteroskedasticity-and-autocorrelation-consistent covariance
estimation.

A Bartlett-kernel HAC estimator with a 12-month bandwidth and a finite-sample correction is used. The 12-month bandwidth allows the covariance estimator to account for residual dependence across one annual cycle.

HAC estimation does not change the fitted coefficients, selected lags or BIC values. It changes the estimated covariance matrix and therefore may affect standard errors, confidence intervals and p-values.

In [157]:
# re-estimate Covariance Using HAC

def refit_models_with_hac(model_results, max_lags=12):
    robust_results = {}

    for subclass, fitted_model in model_results.items():
        robust_results[subclass] = fitted_model.model.fit(
            cov_type="HAC",
            cov_kwds={
                "maxlags": max_lags,
                "kernel": "bartlett",
                "use_correction": True,
            },
            use_t=True,
        )

    return robust_results


hac_max_lags = 12

inference_model_collections = {
    "Symmetric level": symmetric_level_uecm_results,
    "Asymmetric level": asymmetric_level_uecm_results,
    "Symmetric short run": symmetric_short_run_results,
    "Asymmetric short run": asymmetric_short_run_results,
}

robust_model_collections = {
    model_name: refit_models_with_hac(
        model_results,
        max_lags=hac_max_lags,
    )
    for model_name, model_results
    in inference_model_collections.items()
}

print(
    "HAC-adjusted models:",
    sum(
        len(model_results)
        for model_results in robust_model_collections.values()
    ),
)

HAC-adjusted models: 184


In [158]:
# validate the HAC Re-estimation

hac_validation_records = []

for model_name, standard_results in (
    inference_model_collections.items()
):
    robust_results = robust_model_collections[model_name]
    coefficient_differences = []

    for subclass, standard_model in standard_results.items():
        robust_model = robust_results[subclass]

        difference = np.max(
            np.abs(
                np.asarray(standard_model.params)
                - np.asarray(robust_model.params)
            )
        )

        coefficient_differences.append(difference)

    hac_validation_records.append(
        {
            "Model": model_name,
            "Models": len(standard_results),
            "Maximum_Coefficient_Difference": max(
                coefficient_differences
            ),
            "Coefficients_Unchanged": np.allclose(
                coefficient_differences,
                0.0,
                atol=1e-10,
            ),
        }
    )

hac_validation = pd.DataFrame(hac_validation_records)

display(hac_validation)

,Model,Models,Maximum_Coefficient_Difference,Coefficients_Unchanged
0,Symmetric level,46,0.000000,True
1,Asymmetric level,46,0.000000,True
2,Symmetric short run,46,0.000000,True
3,Asymmetric short run,46,0.000000,True


### HAC Re-estimation Validation

All 184 econometric specifications were re-estimated using HAC covariance estimation with a 12-month Bartlett bandwidth.

The coefficient estimates from the HAC-adjusted models are identical to those from the original models. This is expected because HAC estimation changes the estimated covariance matrix rather than the underlying model coefficients.

Consequently, the selected lag structures, fitted values and BIC comparisons remain unchanged. The HAC results are used only to reassess standard errors, confidence intervals and statistical significance.

In [159]:
# extract HAC Adjustment Results

def extract_hac_adjustment_results(
    adjustment_results,
    robust_results,
    significance_level=0.05,
):
    model_mapping = {
        "ARDL": "Symmetric level",
        "NARDL": "Asymmetric level",
    }

    records = []

    for row in adjustment_results.itertuples(index=False):
        model_family = model_mapping[row.Model]
        fitted_model = robust_results[model_family][row.SubclassDescription]

        parameter_names = list(fitted_model.model.exog_names)
        parameter_index = parameter_names.index("Log_CPI.L1")

        coefficients = np.asarray(fitted_model.params, dtype=float)
        standard_errors = np.asarray(fitted_model.bse, dtype=float)
        p_values = np.asarray(fitted_model.pvalues, dtype=float)
        confidence_intervals = np.asarray(
            fitted_model.conf_int(alpha=significance_level),
            dtype=float,
        )

        coefficient = coefficients[parameter_index]
        standard_error = standard_errors[parameter_index]
        p_value = p_values[parameter_index]
        lower_bound, upper_bound = confidence_intervals[parameter_index]

        stable_adjustment = (
            -1 < coefficient < 0
            and p_value < significance_level
        )

        records.append(
            {
                "SubclassDescription": row.SubclassDescription,
                "Model": row.Model,
                "HAC_Adjustment_Coefficient": coefficient,
                "HAC_Standard_Error": standard_error,
                "HAC_P_Value": p_value,
                "HAC_Lower_95pct": lower_bound,
                "HAC_Upper_95pct": upper_bound,
                "HAC_Stable_Adjustment": stable_adjustment,
            }
        )

    return pd.DataFrame(records)


hac_adjustment_results = extract_hac_adjustment_results(
    error_correction_results,
    robust_model_collections,
)

print(
    "HAC adjustment coefficients extracted:",
    len(hac_adjustment_results),
)

HAC adjustment coefficients extracted: 9


In [160]:
# compare Conventional and HAC Inference
adjustment_inference_comparison = (
    error_correction_results[
        [
            "SubclassDescription",
            "Model",
            "Adjustment_Coefficient",
            "Standard_Error",
            "P_Value",
            "Stable_Adjustment",
        ]
    ]
    .rename(
        columns={
            "Standard_Error": "Conventional_Standard_Error",
            "P_Value": "Conventional_P_Value",
            "Stable_Adjustment": "Conventional_Stable_Adjustment",
        }
    )
    .merge(
        hac_adjustment_results,
        on=["SubclassDescription", "Model"],
        how="inner",
        validate="one_to_one",
    )
)

adjustment_inference_comparison["Coefficient_Difference"] = (
    adjustment_inference_comparison["Adjustment_Coefficient"]
    - adjustment_inference_comparison["HAC_Adjustment_Coefficient"]
).abs()

adjustment_inference_comparison["Inference_Changed"] = (
    adjustment_inference_comparison["Conventional_Stable_Adjustment"]
    != adjustment_inference_comparison["HAC_Stable_Adjustment"]
)

display(
    adjustment_inference_comparison[
        [
            "SubclassDescription",
            "Model",
            "Adjustment_Coefficient",
            "Conventional_Standard_Error",
            "HAC_Standard_Error",
            "Conventional_P_Value",
            "HAC_P_Value",
            "Conventional_Stable_Adjustment",
            "HAC_Stable_Adjustment",
            "Inference_Changed",
        ]
    ].sort_values(["Model", "SubclassDescription"])
)

adjustment_summary = (
    adjustment_inference_comparison
    .groupby("Model", as_index=False)
    .agg(
        Models=("SubclassDescription", "size"),
        Conventional_Stable=(
            "Conventional_Stable_Adjustment",
            "sum",
        ),
        HAC_Stable=("HAC_Stable_Adjustment", "sum"),
        Changed_Conclusions=("Inference_Changed", "sum"),
    )
)

display(adjustment_summary)

print(
    "Maximum coefficient difference:",
    adjustment_inference_comparison[
        "Coefficient_Difference"
    ].max(),
)

,SubclassDescription,Model,Adjustment_Coefficient,Conventional_Standard_Error,HAC_Standard_Error,Conventional_P_Value,HAC_P_Value,Conventional_Stable_Adjustment,HAC_Stable_Adjustment,Inference_Changed
0,"Chocolate, cocoa, and cocoa-based food products",ARDL,-0.008389,0.007680,0.008448,0.277874,0.323580,False,False,False
1,"Other vegetables, fresh or chilled",ARDL,-0.066982,0.018813,0.029423,0.000616,0.025391,True,True,False
2,Yoghurt and similar products,ARDL,-0.046384,0.012117,0.011848,0.000250,0.000185,True,True,False
3,Cereals,NARDL,-0.066162,0.020543,0.015666,0.001850,0.000063,True,True,False
4,"Chocolate, cocoa, and cocoa-based food products",NARDL,-0.027135,0.013297,0.009275,0.044496,0.004444,True,True,False
5,"Dates, figs and tropical fruits, fresh",NARDL,-0.227812,0.069081,0.065086,0.001449,0.000759,True,True,False
6,"Fruit-bearing vegetables, fresh or chilled",NARDL,-0.346863,0.089963,0.071885,0.000236,0.000007,True,True,False
7,"Other vegetables, fresh or chilled",NARDL,-0.109172,0.026889,0.033413,0.000113,0.001593,True,True,False
8,Yoghurt and similar products,NARDL,-0.128573,0.032106,0.036687,0.000137,0.000748,True,True,False


,Model,Models,Conventional_Stable,HAC_Stable,Changed_Conclusions
0,ARDL,3,2,2,0
1,NARDL,6,6,6,0


Maximum coefficient difference: 9.71445146547012e-17


### HAC Adjustment Results

The HAC-adjusted inference confirms the original error-correction conclusions.

Eight of the nine bounds-supported models retain a negative and statistically significant adjustment coefficient. These consist of two symmetric ARDL models and all six asymmetric NARDL models.

The symmetric model for chocolate and cocoa-based food products remains the only bounds-supported specification without a statistically significant adjustment mechanism. It is therefore not used for primary long-run interpretation.

Although HAC adjustment changes some standard errors and p-values, it does not change the substantive conclusions regarding equilibrium correction. The remaining eight models provide robust evidence that food prices respond to departures from their estimated long-run relationships.

In [161]:
# calculate HAC Long-Run Effects

def calculate_hac_long_run_effect(
    fitted_model,
    numerator_name,
    scale=1.0,
    significance_level=0.05,
):
    parameter_names = list(fitted_model.model.exog_names)
    parameters = np.asarray(fitted_model.params, dtype=float)
    covariance = np.asarray(fitted_model.cov_params(), dtype=float)

    adjustment_index = parameter_names.index("Log_CPI.L1")
    numerator_index = parameter_names.index(numerator_name)

    adjustment = parameters[adjustment_index]
    numerator = parameters[numerator_index]

    multiplier = -scale * numerator / adjustment

    gradient = np.zeros(len(parameters))
    gradient[adjustment_index] = scale * numerator / adjustment**2
    gradient[numerator_index] = -scale / adjustment

    variance = float(gradient @ covariance @ gradient)
    standard_error = np.sqrt(max(variance, 0.0))

    t_statistic = multiplier / standard_error
    p_value = 2 * stats.t.sf(
        abs(t_statistic),
        df=fitted_model.df_resid,
    )

    critical_value = stats.t.ppf(
        1 - significance_level / 2,
        df=fitted_model.df_resid,
    )

    lower_bound = multiplier - critical_value * standard_error
    upper_bound = multiplier + critical_value * standard_error

    return {
        "HAC_Long_Run_Multiplier": multiplier,
        "HAC_Standard_Error": standard_error,
        "HAC_P_Value": p_value,
        "HAC_Lower_95pct": lower_bound,
        "HAC_Upper_95pct": upper_bound,
    }


effect_specifications = {
    "ARDL": [
        {
            "effect": "Symmetric exchange-rate effect",
            "parameter": "Log_ExchangeRate.L1",
            "scale": 1.0,
            "unit": "Elasticity",
        }
    ],
    "NARDL": [
        {
            "effect": "Depreciation",
            "parameter": (
                "ExchangeRate_Positive_Cumulative_Pct.L1"
            ),
            "scale": 100.0,
            "unit": "Percent CPI response",
        },
        {
            "effect": "Appreciation component",
            "parameter": (
                "ExchangeRate_Negative_Cumulative_Pct.L1"
            ),
            "scale": 100.0,
            "unit": "Percent CPI response",
        },
    ],
}

hac_long_run_records = []

eligible_adjustment_results = adjustment_inference_comparison.loc[
    adjustment_inference_comparison["HAC_Stable_Adjustment"]
]

for row in eligible_adjustment_results.itertuples(index=False):
    model_family = {
        "ARDL": "Symmetric level",
        "NARDL": "Asymmetric level",
    }[row.Model]

    fitted_model = robust_model_collections[
        model_family
    ][row.SubclassDescription]

    for specification in effect_specifications[row.Model]:
        effect_result = calculate_hac_long_run_effect(
            fitted_model=fitted_model,
            numerator_name=specification["parameter"],
            scale=specification["scale"],
        )

        hac_long_run_records.append(
            {
                "SubclassDescription": row.SubclassDescription,
                "Model": row.Model,
                "Effect": specification["effect"],
                "Unit": specification["unit"],
                **effect_result,
            }
        )

hac_long_run_effects = pd.DataFrame(hac_long_run_records)

print(
    "HAC long-run effects extracted:",
    len(hac_long_run_effects),
)

display(hac_long_run_effects)

HAC long-run effects extracted: 14


,SubclassDescription,Model,Effect,Unit,HAC_Long_Run_Multiplier,HAC_Standard_Error,HAC_P_Value,HAC_Lower_95pct,HAC_Upper_95pct
0,"Other vegetables, fresh or chilled",ARDL,Symmetric exchange-rate effect,Elasticity,1.404119,0.226116,0.000000,0.954384,1.853854
1,Yoghurt and similar products,ARDL,Symmetric exchange-rate effect,Elasticity,1.283947,0.167883,0.000000,0.950035,1.617860
2,Cereals,NARDL,Depreciation,Percent CPI response,1.330943,0.450046,0.004078,0.435324,2.226562
3,Cereals,NARDL,Appreciation component,Percent CPI response,1.413603,0.576771,0.016431,0.265791,2.561415
4,"Chocolate, cocoa, and cocoa-based food products",NARDL,Depreciation,Percent CPI response,0.957384,0.511100,0.064606,-0.059357,1.974124
5,"Chocolate, cocoa, and cocoa-based food products",NARDL,Appreciation component,Percent CPI response,0.465273,0.682479,0.497324,-0.892395,1.822941
6,"Dates, figs and tropical fruits, fresh",NARDL,Depreciation,Percent CPI response,0.313512,0.189371,0.101683,-0.063276,0.690300
7,"Dates, figs and tropical fruits, fresh",NARDL,Appreciation component,Percent CPI response,0.030170,0.269769,0.911228,-0.506585,0.566926
8,"Fruit-bearing vegetables, fresh or chilled",NARDL,Depreciation,Percent CPI response,0.258992,0.141529,0.071078,-0.022771,0.540755
9,"Fruit-bearing vegetables, fresh or chilled",NARDL,Appreciation component,Percent CPI response,-0.039340,0.193053,0.839056,-0.423680,0.344999


In [162]:
# compare Long-Run Inference

long_run_inference_comparison = (
    long_run_effects[
        [
            "SubclassDescription",
            "Model",
            "Effect",
            "Long_Run_Multiplier",
            "Standard_Error",
            "P_Value",
        ]
    ]
    .rename(
        columns={
            "Standard_Error": "Conventional_Standard_Error",
            "P_Value": "Conventional_P_Value",
        }
    )
    .merge(
        hac_long_run_effects.drop(columns="Unit"),
        on=["SubclassDescription", "Model", "Effect"],
        how="inner",
        validate="one_to_one",
    )
)

long_run_inference_comparison["Multiplier_Difference"] = (
    long_run_inference_comparison["Long_Run_Multiplier"]
    - long_run_inference_comparison["HAC_Long_Run_Multiplier"]
).abs()

long_run_inference_comparison["Conventional_Significant"] = (
    long_run_inference_comparison["Conventional_P_Value"] < 0.05
)

long_run_inference_comparison["HAC_Significant"] = (
    long_run_inference_comparison["HAC_P_Value"] < 0.05
)

long_run_inference_comparison["Inference_Changed"] = (
    long_run_inference_comparison["Conventional_Significant"]
    != long_run_inference_comparison["HAC_Significant"]
)

display(
    long_run_inference_comparison[
        [
            "SubclassDescription",
            "Model",
            "Effect",
            "Long_Run_Multiplier",
            "Conventional_Standard_Error",
            "HAC_Standard_Error",
            "Conventional_P_Value",
            "HAC_P_Value",
            "Conventional_Significant",
            "HAC_Significant",
            "Inference_Changed",
        ]
    ].sort_values(
        ["Model", "SubclassDescription", "Effect"]
    )
)

long_run_inference_summary = pd.DataFrame(
    {
        "Value": [
            len(long_run_inference_comparison),
            long_run_inference_comparison[
                "Conventional_Significant"
            ].sum(),
            long_run_inference_comparison[
                "HAC_Significant"
            ].sum(),
            long_run_inference_comparison[
                "Inference_Changed"
            ].sum(),
            long_run_inference_comparison[
                "Multiplier_Difference"
            ].max(),
        ]
    },
    index=[
        "Long-run effects",
        "Conventionally significant",
        "HAC significant",
        "Changed conclusions",
        "Maximum multiplier difference",
    ],
)

display(long_run_inference_summary)

,SubclassDescription,Model,Effect,Long_Run_Multiplier,Conventional_Standard_Error,HAC_Standard_Error,Conventional_P_Value,HAC_P_Value,Conventional_Significant,HAC_Significant,Inference_Changed
0,"Other vegetables, fresh or chilled",ARDL,Symmetric exchange-rate effect,1.404119,0.258553,0.226116,0.000000,0.000000,True,True,False
1,Yoghurt and similar products,ARDL,Symmetric exchange-rate effect,1.283947,0.173339,0.167883,0.000000,0.000000,True,True,False
2,Cereals,NARDL,Appreciation component,1.413603,0.586010,0.576771,0.015854,0.016431,True,True,False
3,Cereals,NARDL,Depreciation,1.330943,0.427525,0.450046,0.001851,0.004078,True,True,False
4,"Chocolate, cocoa, and cocoa-based food products",NARDL,Appreciation component,0.465273,0.671379,0.682479,0.488302,0.497324,False,False,False
5,"Chocolate, cocoa, and cocoa-based food products",NARDL,Depreciation,0.957384,0.554710,0.511100,0.084362,0.064606,False,False,False
6,"Dates, figs and tropical fruits, fresh",NARDL,Appreciation component,0.030170,0.310494,0.269769,0.922592,0.911228,False,False,False
7,"Dates, figs and tropical fruits, fresh",NARDL,Depreciation,0.313512,0.229835,0.189371,0.172545,0.101683,False,False,False
8,"Fruit-bearing vegetables, fresh or chilled",NARDL,Appreciation component,-0.039340,0.188328,0.193053,0.834532,0.839056,False,False,False
9,"Fruit-bearing vegetables, fresh or chilled",NARDL,Depreciation,0.258992,0.138561,0.141529,0.061602,0.071078,False,False,False


,Value
Long-run effects,14.000000
Conventionally significant,6.000000
HAC significant,7.000000
Changed conclusions,1.000000
Maximum multiplier difference,0.000000


### Interpretation of HAC Long-Run Effects

HAC adjustment leaves all 14 long-run multiplier estimates unchanged but changes the statistical inference attached to them.

Seven effects are statistically significant under HAC inference, compared with six under conventional inference. The only changed conclusion concerns the appreciation component for yoghurt and similar products, which becomes significant at the 5% level.

The symmetric exchange-rate effects for other fresh vegetables and yoghurt remain strongly significant. Within the asymmetric models, both exchange-rate components are significant for cereals and yoghurt, while only the depreciation component is significant for other fresh vegetables.

The appreciation variable is a signed negative cumulative component. A positive multiplier therefore means that an appreciation, represented by a decline in this component, is associated with a reduction in food prices.

Individual coefficient significance does not establish asymmetry. A separate test is required to determine whether the depreciation and appreciation multipliers differ significantly from each other.